https://dev.to/lsys/making-coefficient-plots-in-python-using-forestplot-7i7

In [ ]:
pip install forestplot pandas

In [ ]:
# @title Geração dos Dados Fictícios (Simulando projeto LLM)
import pandas as pd

dados_simulados = pd.DataFrame({
    "par": ["GPT-4 − GPT-3.5", "Claude-3 − GPT-4", "Llama-3 − GPT-3.5", "Gemini − Claude-3"],
    "diferenca_media": [0.45, -0.05, 0.12, 0.02],
    "ic_inf": [0.20, -0.15, -0.08, -0.10],
    "ic_sup": [0.70, 0.05, 0.32, 0.14],
    "p_rope": [0.01, 0.96, 0.40, 0.98],
    "classificacao": ["superior", "equivalente", "incerto", "equivalente"]
})
rope = 0.15


In [ ]:
# @title Option 1: Using the Dedicated forestplot Library
import forestplot as fp

# fp requires specific columns. It doesn't support coloring by classification natively
# nor does it support a shaded ROPE band natively.
fp.forestplot(
    dados_simulados,
    estimate="diferenca_media",
    ll="ic_inf",
    hl="ic_sup",
    varlabel="par",
    annote=["diferenca_media", "ic_inf", "ic_sup", "p_rope"],
    annoteheaders=["Diferença Média", "IC Inf", "IC Sup", "P(equiv)"],
    xlabel="Diferença (Positivo = Primeiro Melhor)",
    ci_report=False,
)


In [ ]:
# @title Option 2: Using Pure matplotlib
import matplotlib.pyplot as plt

pares = dados_simulados["par"].tolist()
difs = dados_simulados["diferenca_media"].tolist()
infs = dados_simulados["ic_inf"].tolist()
sups = dados_simulados["ic_sup"].tolist()
y_pos = range(len(pares))

fig, ax = plt.subplots(figsize=(8, 4))
ax.axvspan(-rope, rope, color="#dfe9f0", zorder=0, label=f"ROPE (± {rope})")
ax.axvline(x=0.0, color="gray", linestyle="--", linewidth=1.2)

for i in range(len(pares)):
    left_err = difs[i] - infs[i]
    right_err = sups[i] - difs[i]
    ax.errorbar(
        x=difs[i],
        y=y_pos[i],
        xerr=[[left_err], [right_err]],
        fmt="o",
        color="black",
        markersize=7,
        capsize=4,
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(pares)
ax.invert_yaxis()
ax.set_xlabel("Diferença (Positivo = Primeiro Melhor)")
ax.legend()
plt.show()


Pacote personalizado para o baycomp: Medindo as diferenças

In [ ]:
# @title Option 3: Usando nosso pacote do projeto
import sys
sys.path.append('../src')
from util_est_bayesiana import grafico_diferencas
import pandas as pd

# O gráfico do pacote espera um DataFrame (matriz de pares) com attributos rope e limiar,
# e formato de dicionário por linha. Vamos simular isso empacotando os dados simulados:
dados_simulados.attrs['rope'] = rope
dados_simulados.attrs['limiar'] = 0.95

# Como a função _pares_nao_ordenados no pacote espera colunas específicas
# (linha, coluna, classificacao, ic_inf, ic_sup, diferenca_media, p_rope),
# vamos ajustar nosso dataset fictício para bater com o esperado:
dados_simulados['linha'] = [p.split(' − ')[0] for p in dados_simulados['par']]
dados_simulados['coluna'] = [p.split(' − ')[1] for p in dados_simulados['par']]

figura, caminho = grafico_diferencas(dados_simulados, titulo="Demonstração do Pacote")
plt.show()
